# Демо 3. RAG на LangChain і оцінювання

У демо 2 ми написали RAG-пайплайн вручну. Тут збираємо той самий пайплайн із готових компонентів
[LangChain](https://python.langchain.com/), додаємо два способи покращити пошук — **гібридний пошук**
і **переранжування** — і, головне, **вимірюємо** якість системи на наборі питань із відомими відповідями.

Ретрив і генерацію оцінюємо окремо:
- **ретрив** — чи потрапляє фрагмент із потрібного документа до знайдених (hit rate@k, MRR);
- **генерація** — чи правильна відповідь, чи посилається вона на правильне джерело
  і чи відмовляється система відповідати, коли відповіді в документах немає.

_Для цього ноутбука потрібен GPU ._

In [1]:
# LangChain розбитий на пакети: core — базові абстракції, text-splitters — розбиття тексту,
# community — інтеграції (FAISS, BM25), huggingface — моделі з Hugging Face
!pip install -q "sentence-transformers>=5" pypdf langchain-core langchain-text-splitters langchain-community langchain-huggingface faiss-cpu rank_bm25

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

/tmp/ipykernel_2372/1402898701.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Пристрій: cuda


## 1. Документи

Ті самі 14 документів, що в демо 1 і 2. LangChain має готові завантажувачі PDF, але колонтитули
й скани все одно доводиться обробляти самостійно, тому текст витягуємо так само, як раніше,
а потім загортаємо кожен документ в обʼєкт `Document`: текст плюс метадані.

In [3]:
# @title Завантаження й очищення документів (як у демо 1)
BASE_URL = "https://docs.ztu.edu.ua/"
DOCUMENTS = [
    ("275", "Положення про оцінювання результатів навчання (чинна версія 4.0)", {"mdocs-file": 275}),
    ("275-v3.1", "Положення про оцінювання результатів навчання (стара версія 3.1)",
     {"mdocs-version": "polozhennja pro ocinjuvannja znan yekts_nove.pdf-v3.1", "mdocs-file": 275, "mdocs-url": "false"}),
    ("277", "Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін", {"mdocs-file": 277}),
    ("291", "Положення про проведення практики здобувачів вищої освіти", {"mdocs-file": 291}),
    ("340", "Правила призначення академічних стипендій", {"mdocs-file": 340}),
    ("344", "Положення про стипендію ректора", {"mdocs-file": 344}),
    ("348", "Положення про користування гуртожитками", {"mdocs-file": 348}),
    ("1219", "Правила внутрішнього розпорядку в студентських гуртожитках", {"mdocs-file": 1219}),
    ("1199", "Кодекс академічної доброчесності", {"mdocs-file": 1199}),
    ("4389", "Положення про студентського омбудсмана", {"mdocs-file": 4389}),
    ("336", "Положення про старосту академічної групи", {"mdocs-file": 336}),
    ("3320", "ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року", {"mdocs-file": 3320}),
    ("3329", "ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року", {"mdocs-file": 3329}),
    ("283", "Положення про переведення здобувачів вищої освіти", {"mdocs-file": 283}),
]


PDF_DIR = Path("ztu_pdfs")
PDF_DIR.mkdir(exist_ok=True)


def fetch(params, attempts=3):
    # Сайт іноді обриває зʼєднання, тому пробуємо кілька разів
    for attempt in range(attempts):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(5)


def download_pdf(doc_id, params):
    # Зберігаємо PDF на диск, щоб під час повторного запуску не завантажувати їх знову
    path = PDF_DIR / f"{doc_id}.pdf"
    if not path.exists():
        path.write_bytes(fetch(params))
    return PdfReader(path)


HEADER_LINE = re.compile(
    r"^(Житомирська|політехніка|Міністерство освіти.*|Державний університет «Житомирська політехніка»|"
    r"Система управління якістю.*|П[\s\-–]*[CС]?[АA]?[ГG]?[\s\d.\-–]*|[\d.\-–\s/_]+|Випуск.*)$",
    re.IGNORECASE,
)
HEADER_LINES = 12


def page_text(page):
    lines = (page.extract_text() or "").split("\n")
    head = [line for line in lines[:HEADER_LINES] if not HEADER_LINE.match(line.strip())]
    return " ".join(head + lines[HEADER_LINES:])


MIN_CHARS_PER_PAGE = 100


def document_text(reader):
    return re.sub(r"\s+", " ", " ".join(page_text(page) for page in reader.pages)).strip()


def is_scan(text, reader):
    return len(text) < MIN_CHARS_PER_PAGE * len(reader.pages)


documents = {}
for doc_id, title, params in DOCUMENTS:
    reader = download_pdf(doc_id, params)
    text = document_text(reader)
    if is_scan(text, reader):
        print(f"{doc_id}: текстового шару немає, потрібне OCR — пропущено")
        continue
    documents[doc_id] = {"title": title, "text": text}
    print(f"{doc_id}: {len(reader.pages)} сторінок, {len(text)} символів — {title}")

275: 23 сторінок, 24944 символів — Положення про оцінювання результатів навчання (чинна версія 4.0)


275-v3.1: 13 сторінок, 10505 символів — Положення про оцінювання результатів навчання (стара версія 3.1)


277: 16 сторінок, 15945 символів — Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін


291: 17 сторінок, 14603 символів — Положення про проведення практики здобувачів вищої освіти


340: 30 сторінок, 35706 символів — Правила призначення академічних стипендій


344: 10 сторінок, 2455 символів — Положення про стипендію ректора


... (3 outputs left out)

4389: 18 сторінок, 20685 символів — Положення про студентського омбудсмана


336: 8 сторінок, 3560 символів — Положення про старосту академічної групи


3320: 16 сторінок, 31670 символів — ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року


3329: 17 сторінок, 31193 символів — ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року
283: текстового шару немає, потрібне OCR — пропущено


In [4]:
docs = [
    Document(page_content=doc["text"], metadata={"doc_id": doc_id, "title": doc["title"]})
    for doc_id, doc in documents.items()
]
print(docs[0].metadata, docs[0].page_content[:200])

{'doc_id': '275', 'title': 'Положення про оцінювання результатів навчання (чинна версія 4.0)'} ЗАТВЕРДЖЕНО Наказ Державного університету «Житомирська політехніка» 15 вересня 2025 р. № 327/од Ректор Віктор ЄВДОКИМОВ ПОЛОЖЕННЯ про оцінювання результатів навчання здобувачів вищої освіти у Державно


## 2. Розбиття на фрагменти

`RecursiveCharacterTextSplitter` пробує різати текст за першим роздільником зі списку; якщо шматок
усе ще задовгий, переходить до наступного роздільника. Наш список: кінець речення, крапка з комою,
пробіл і, в крайньому разі, будь-який символ. Метадані документа копіюються в кожен фрагмент.
Як і раніше, дописуємо на початок фрагмента назву документа.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, separators=[". ", "; ", " ", ""], keep_separator="end"
)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    chunk.page_content = f"{chunk.metadata['title']}. {chunk.page_content}"

print(f"Фрагментів: {len(chunks)}")
print(chunks[100].metadata)
print(chunks[100].page_content)

Фрагментів: 385
{'doc_id': '340', 'title': 'Правила призначення академічних стипендій'}
Правила призначення академічних стипендій. 1.9. Студенти, аспіранти (докторанти), які реалізують право на академічну мобільність і зберігають статус здобувача вищої освіти (наукового ступеня) за денною формою навчання (навчаються з відривом від виробництва) у Житомирській політехніці, протягом строку навчання за програмою академічної мобільності в іншому навчальному закладі поза межами України отримують стипендію, призначену відповідно до цих Правил, у разі, коли умовами договору про навчання за програмою академічної мобільності, укладеного ними з Житомирською політехнікою, регулярна безповоротна фіксована фінансова підтримка у грошовій формі протягом всього строку навчання за програмою академічної мобільності в іншому навчальному закладі не передбачена або передбачена у розмірі, що у перерахунку на національну грошову одиницю на дату укладення договору про академічну мобільність є меншою за розмір 

## 3. Векторне сховище

`HuggingFaceEmbeddings` — обгортка LangChain над sentence-transformers. Для документів і запитів
задаємо різні параметри: запит кодується з промптом `query`, як у демо 1.
[FAISS](https://github.com/facebookresearch/faiss) — бібліотека векторного пошуку від Meta;
для нормалізованих векторів обираємо скалярний добуток як міру подібності.

In [6]:
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device, "model_kwargs": {"dtype": dtype}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    query_encode_kwargs={"normalize_embeddings": True, "prompt_name": "query"},
)
vector_store = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

for doc, score in vector_store.similarity_search_with_score("Коли обирають старосту на першому курсі?", k=3):
    print(f"{score:.2f}  [{doc.metadata['doc_id']}] {doc.page_content[:120]}…")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


0.60  [336] Положення про старосту академічної групи. У своїй роботі староста академічної групи співпрацює з куратором та ментором г…
0.53  [336] Положення про старосту академічної групи. До виборів обов’язки старости групи може виконувати один із студентів групи за…
0.50  [4389] Положення про студентського омбудсмана. 6.4. Студентський омбудсман взаємодіє з Молодіжною радою при Житомирській обласн…


## 4. Лексичний пошук BM25

BM25 — класичний алгоритм пошуку за збігом слів: фрагмент отримує тим більшу оцінку, чим більше в ньому
слів запиту і чим рідше ці слова трапляються в інших фрагментах. BM25 не розуміє синонімів і,
без додаткової обробки, різних форм того самого слова («гуртожиток» і «гуртожитку» для нього різні слова),
зате точно знаходить рідкісні слова, номери й коди.

In [7]:
CANDIDATES = 20


def tokenize(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Retriever.from_documents(chunks, k=CANDIDATES, preprocess_func=tokenize)


def vector_search(query, k):
    return vector_store.similarity_search(query, k=k)


def bm25_search(query, k):
    return bm25.invoke(query)[:k]


def show(search, query, k=3):
    print(f"{search.__name__}: {query}")
    for doc in search(query, k):
        print(f"  [{doc.metadata['doc_id']}] {doc.page_content[:110]}…")
    print()


for query in ["До якої дати потрібно оплатити гуртожиток за перший семестр?",
              "Яку допомогу отримує студент в академічній відпустці за медичними показаннями?"]:
    show(vector_search, query)
    show(bm25_search, query)

vector_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 5.2. Студент вносить авансовий платіж оплати за 1 семестр (5 місяців)…
  [348] Положення про користування гуртожитками. Мешканці гуртожитків Житомирської політехніки зобов’язані: - знати й …
  [348] Положення про користування гуртожитками. Це є єдиною підставою для поселення і проживання студента у вказаному…

bm25_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 3.9. Здобувачам освіти, які поселяються до гуртожитку Житомирської по…
  [340] Правила призначення академічних стипендій. Цей показник встановлюється у відсотках (у діапазоні від 40 до 45) …
  [340] Правила призначення академічних стипендій. Рішенням ректора Житомирської політехніки таким особам може встанов…

vector_search: Яку допомогу отримує студент в академічній відпустці за медичними показаннями?
  [340] Правила призначення академ

Фрагмент з відповіддю на перше питання містить слова «авансовий платіж оплати… до 1 вересня»,
а не «оплатити»: семантичний пошук знаходить його, а BM25 — лише серед нижчих результатів.
У другому питанні рідкісний вираз «за медичними показаннями» дослівно є у фрагменті з відповіддю,
і BM25 ставить цей фрагмент першим, а семантичний пошук — нижче.

## 5. Гібридний пошук

Два списки результатів обʼєднуємо методом **reciprocal rank fusion** (RRF): фрагмент на позиції $r$
у кожному списку отримує $\frac{1}{60 + r}$ балів, бали з обох списків додаються.
RRF використовує лише позиції, тому не потребує зведення оцінок BM25 і косинусної подібності до однієї шкали.

In [8]:
def rrf(rankings, c=60):
    scores, by_content = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (c + rank)
            by_content[doc.page_content] = doc
    return [by_content[content] for content in sorted(scores, key=scores.get, reverse=True)]


def hybrid_search(query, k):
    return rrf([vector_search(query, CANDIDATES), bm25_search(query, CANDIDATES)])[:k]


show(hybrid_search, "Якому діапазону балів відповідає оцінка FX?")

hybrid_search: Якому діапазону балів відповідає оцінка FX?


  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Під час поточного контролю оцінюванню можуть…



## 6. Переранжування

Модель ембедингів — **bi-encoder**: запит і фрагмент кодуються окремо, тому ембединги фрагментів можна порахувати
заздалегідь. **Cross-encoder** отримує запит і фрагмент разом і оцінює їхню відповідність точніше,
але для кожної пари потрібен окремий прохід моделі. Тому cross-encoder застосовують лише до кількох десятків
кандидатів, знайдених швидким пошуком.
Використовуємо багатомовну модель [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3).

In [9]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})


def rerank_search(query, k):
    candidates = hybrid_search(query, CANDIDATES)
    scores = reranker.predict([(query, doc.page_content) for doc in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]


show(rerank_search, "Якому діапазону балів відповідає оцінка FX?")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

rerank_search: Якому діапазону балів відповідає оцінка FX?


  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Значення вагового коефіцієнта для кожного ви…



## 7. Бенчмарк

Набір питань складено **до** налаштування системи. Для кожного питання вказано документи, у яких є відповідь,
регулярний вираз, за яким можна впізнати фрагмент із відповіддю, та еталонну відповідь. Три питання не мають відповіді в документах: на них система має відмовитися відповідати.

In [10]:
BENCHMARK = [
    ("До якої дати потрібно оплатити гуртожиток за перший семестр?", ["348"], r"1 вересня", "До 1 вересня поточного року."),
    ("До якої дати оплачують гуртожиток за другий семестр?", ["348"], r"1 лютого", "До 1 лютого поточного року."),
    ("З якої до якої години в гуртожитку має бути тиша?", ["1219"], r"23:00 години і до 7:00", "З 23:00 до 7:00."),
    ("У який час гості можуть перебувати в гуртожитку?", ["1219", "348"], r"0?8:00 до 22:00", "З 8:00 до 22:00."),
    ("Чи можна тримати в гуртожитку домашню тварину?", ["1219", "348"], r"утримувати тварин", "Ні, утримувати тварин у гуртожитку заборонено."),
    ("На який строк укладають договір найму житла в гуртожитку?", ["348"], r"терміном на один навчальний рік", "На один навчальний рік."),
    ("За який час потрібно звільнити гуртожиток після відрахування?", ["1219"], r"протягом двох тижнів", "Протягом двох тижнів від дня виходу наказу."),
    ("Коли обирають старосту на першому курсі?", ["336"], r"після двох тижнів", "На початку навчального року, після двох тижнів навчального процесу."),
    ("На який строк обирається староста групи?", ["336"], r"на весь період навчання", "На весь період навчання."),
    ("Скільки балів потрібно для оцінки «добре» B?", ["275"], r"82 ?– ?89", "82–89 балів."),
    ("Якому діапазону балів відповідає оцінка FX?", ["275"], r"35 ?– ?59", "35–59 балів."),
    ("Яку частку кредитів становлять вибіркові дисципліни?", ["277"], r"25 відсотків",
     "Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням)."),
    ("До якої дати викладачі вносять анотації вибіркових дисциплін?", ["277"], r"30 квітня", "До 30 квітня."),
    ("Як часто виплачують стипендію?", ["340"], r"один раз на місяць", "Один раз на місяць."),
    ("За яким показником формують рейтинг для призначення академічної стипендії?", ["340"], r"середнім балом", "За середнім балом семестрового контролю."),
    ("Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної",
     "Щомісячну допомогу в розмірі 50 відсотків мінімальної академічної стипендії."),
    ("Для кого встановлюється стипендія ректора?", ["344"], r"встановлюється для студентів денної",
     "Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності."),
    ("На який строк обирають студентського омбудсмана?", ["4389"], r"терміном на 1 рік", "На 1 рік."),
    ("До якої дати проводять вибори студентського омбудсмана?", ["4389"], r"до 01 жовтня", "До 1 жовтня кожного навчального року."),
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат –", "Оприлюднення власних раніше опублікованих результатів без зазначення джерела попереднього оприлюднення."),
    ("Що таке фабрикація в академічній діяльності?", ["1199"], r"Фабрикація –", "Вигадування даних чи фактів про результати власної академічної діяльності."),
    ("Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС?", ["3320", "3329"], r"240 кредит", "240 кредитів ЄКТС."),
    ("Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?", ["3320"], r"спеціальн\w* F2", "F2."),
    ("Скільки коштує обід у студентській їдальні?", [], None, None),
    ("Який графік роботи бібліотеки університету?", [], None, None),
    ("Скільки коштує рік навчання на контракті за спеціальністю «Інженерія програмного забезпечення»?", [], None, None),
]
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")

Питань: 26, з них без відповіді в документах: 3


### Метрики ретриву

Фрагмент вважаємо правильним, якщо він походить із правильного документа і містить вираз-доказ.
Перевіряти лише документ замало: у довгому документі десятки фрагментів, і більшість із них відповіді не містять.

- **hit rate@k** — частка питань, для яких правильний фрагмент є серед `k` знайдених;
- **MRR** (mean reciprocal rank) — середнє за питаннями значення $\frac{1}{r}$, де $r$ — позиція першого
  правильного фрагмента; якщо його немає серед перших 10, замість $\frac{1}{r}$ береться 0. MRR враховує не лише те, чи знайдено фрагмент, а й те, наскільки високо.

Питання без відповіді в ретриві не оцінюємо: правильного фрагмента для них немає.

In [11]:
def first_hit_rank(results, gold, evidence):
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content):
            return rank
    return None


def hit_rate(ranks, k):
    return np.mean([rank is not None and rank <= k for rank in ranks])


def mrr(ranks):
    return np.mean([1 / rank if rank else 0 for rank in ranks])


def retrieval_metrics(search):
    ranks = [first_hit_rank(search(question, 10), gold, evidence) for question, gold, evidence in answerable]
    return {"метод": search.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}


pd.DataFrame([retrieval_metrics(search) for search in [vector_search, bm25_search, hybrid_search, rerank_search]]).round(2)

,метод,hit@1,hit@5,MRR
0,vector_search,0.61,0.87,0.74
1,bm25_search,0.52,0.65,0.59
2,hybrid_search,0.61,0.91,0.76
3,rerank_search,0.83,0.96,0.90


In [12]:
additional = [
    ("Які дії в університеті вважаються фальсифікацією?", ["1199"], r"Фальсифікація\s*[–—-]",
     "Свідома зміна чи модифікація вже наявних даних, що стосуються академічної діяльності."),
    ("Коли ввечері мешканці мають припинити шуміти в гуртожитку?", ["1219"], r"23:00 години і до 7:00", "З 23:00."),
    ("Де знайти ціни на абонемент до університетського басейну?", [], None, None),
]
methods = [vector_search, bm25_search, hybrid_search, rerank_search]
before = pd.DataFrame([retrieval_metrics(search) for search in methods])
BENCHMARK += additional
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
after = pd.DataFrame([retrieval_metrics(search) for search in methods])
display(pd.concat({"до": before.set_index("метод"), "після": after.set_index("метод")}, axis=1).round(3))
print(f"RRF для позицій 2 і 5: {1 / (60 + 2) + 1 / (60 + 5):.8f}. Повний корпус: {len(chunks)} пар на запит.")

def comparable_rank(search, case):
    q, gold, evidence, _ = case
    return first_hit_rank(search(q, 10), gold, evidence) or 11

comparisons = [(comparable_rank(bm25_search, case), comparable_rank(vector_search, case), case[0])
               for case in BENCHMARK if case[1]]
for label, candidates in [
    ("BM25 вище", [item for item in comparisons if item[0] < item[1]]),
    ("Семантичний вище", [item for item in comparisons if item[1] < item[0]]),
]:
    print(label, max(candidates, key=lambda item: abs(item[0] - item[1]), default="прикладу не знайдено"))
print("Ранг 11 означає відсутність доказу в першій десятці.")

до               після             
               hit@1  hit@5    MRR hit@1 hit@5    MRR
метод                                                
vector_search  0.609  0.870  0.738  0.60  0.84  0.726
bm25_search    0.522  0.652  0.591  0.48  0.64  0.564
hybrid_search  0.609  0.913  0.757  0.60  0.88  0.736
rerank_search  0.826  0.957  0.897  0.84  0.96  0.905

RRF для позицій 2 і 5: 0.03151365. Повний корпус: 385 пар на запит.


BM25 вище (1, 6, 'Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?')
Семантичний вище (11, 1, 'Як часто виплачують стипендію?')
Ранг 11 означає відсутність доказу в першій десятці.


На українських текстах BM25 сам по собі зазвичай поступається семантичному пошуку: слова мають багато форм,
а питання рідко повторюють формулювання документа. Переранжування найсильніше впливає на hit@1:
cross-encoder бачить запит і фрагмент разом і частіше ставить правильний фрагмент на перше місце.
Висновки варто робити обережно: на 23 питаннях одне питання — це понад 4 відсоткові пункти.

## 8. RAG-ланцюжок на LangChain

Генеративна модель та сама, що в демо 2. Загортаємо її в `HuggingFacePipeline`, а `ChatHuggingFace`
застосовує до повідомлень шаблон чату моделі.

In [13]:
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)

generator = pipeline(
    "text-generation", model=llm, tokenizer=tokenizer,
    max_new_tokens=300, do_sample=False, temperature=None, top_p=None, top_k=None, return_full_text=False,
)
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator), tokenizer=tokenizer)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


Ланцюжок записуємо мовою LCEL (LangChain Expression Language): оператор `|` передає вихід одного кроку
на вхід наступного. Ланцюжок повертає і відповідь, і знайдені фрагменти: вони потрібні для перевірки цитувань.

In [14]:
SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання студентів Житомирської політехніки. "
    "Відповідай лише на основі наданих фрагментів документів. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Якщо у фрагментах немає відповіді на питання, відповідай лише: «У наданих документах немає відповіді.» "
    "Відповідай українською мовою, коротко."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Фрагменти документів:\n\n{context}\n\nПитання: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{n}] {doc.page_content}" for n, doc in enumerate(docs, start=1))


retriever = RunnableLambda(lambda question: rerank_search(question, 5))
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | chat
    | StrOutputParser()
)
rag_chain = {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(answer=answer_chain)

result = rag_chain.invoke("Чи можна тримати в гуртожитку домашню тварину?")
print(result["answer"])
for n, doc in enumerate(result["docs"], start=1):
    print(f"  [{n}] {doc.metadata['doc_id']}: {doc.metadata['title']}")

`generation_config` default values have been modified to match model-specific defaults: {'do_sample': True}. If this is not desired, please set these values explicitly.


Ні, в гуртожитку домашню тварину тримати заборонено. [1]
  [1] 348: Положення про користування гуртожитками
  [2] 348: Положення про користування гуртожитками
  [3] 348: Положення про користування гуртожитками
  [4] 1219: Правила внутрішнього розпорядку в студентських гуртожитках
  [5] 1219: Правила внутрішнього розпорядку в студентських гуртожитках


## 9. Оцінювання відповідей

Для кожного питання бенчмарку перевіряємо:
- **правильність**: для питань із відповіддю — чи збігається відповідь з еталонною; для питань без відповіді — чи відмовилася система;
- **цитування**: чи є серед фрагментів, на які посилається відповідь, фрагмент із правильного документа.

Правильність вільної відповіді важко перевірити правилом: «до 1 вересня» і «не пізніше першого вересня» — одна відповідь.
Тому використовуємо підхід **LLM як суддя**: інша (тут — та сама) модель порівнює відповідь з еталоном.
Суддя теж помиляється, тому його вердикти варто вибірково перевіряти вручну.

In [15]:
REFUSAL = "немає відповіді"
JUDGE_PROMPT = (
    "Порівняй відповідь системи з еталонною відповіддю на питання.\n\n"
    "Питання: {question}\nЕталонна відповідь: {reference}\nВідповідь системи: {answer}\n\n"
    "Чи містить відповідь системи ту саму інформацію, що й еталонна, без суперечностей? "
    "Відповідай одним словом: «так» або «ні»."
)


def judge(question, reference, answer):
    verdict = chat.invoke(JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)).content
    return verdict.strip().lower().startswith("так")


def cited_doc_ids(answer, retrieved):
    # Номери на зразок [2] у відповіді перетворюємо на ідентифікатори документів
    numbers = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [retrieved[n - 1] for n in numbers if 0 < n <= len(retrieved)]


def evaluate(question, gold, reference):
    result = rag_chain.invoke(question)
    answer = result["answer"]
    retrieved = [doc.metadata["doc_id"] for doc in result["docs"]]
    if gold:
        correct = judge(question, reference, answer)
        citation_ok = any(doc_id in gold for doc_id in cited_doc_ids(answer, retrieved))
    else:
        correct = REFUSAL in answer.lower()
        citation_ok = None
    return {"питання": question, "правильно": correct, "цитування": citation_ok,
            "документи": gold, "знайдено": retrieved, "відповідь": answer}


evaluation = pd.DataFrame([evaluate(question, gold, reference) for question, gold, _, reference in BENCHMARK])
has_answer = evaluation["цитування"].notna()
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%}")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%}")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Правильних відповідей на питання з відповіддю: 92%
Коректних цитувань: 96%
Правильних відмов на питання без відповіді: 100%


## 10. Аналіз відмов

Найкорисніша частина оцінювання — не підсумкове число, а розбір помилок. Для кожної неправильної відповіді
дивимося, чи був серед знайдених фрагментів правильний документ:
- якщо **не був** — це промах ретриву, і виправляти треба пошук (розбиття, модель ембедингів, гібридний пошук);
- якщо **був** — модель не скористалася контекстом або скористалася неправильно, і виправляти треба промпт чи модель.

In [16]:
def retrieval_verdict(row):
    if not row["документи"] or any(doc_id in row["документи"] for doc_id in row["знайдено"]):
        return "ретрив знайшов"
    return "промах ретриву"


def describe_failure(row):
    print(f"Питання: {row['питання']}")
    print(f"  правильний документ: {row['документи']}, знайдено: {row['знайдено']} — {retrieval_verdict(row)}")
    print(f"  правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"  відповідь: {row['відповідь']}\n")


failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
for _, row in failures.iterrows():
    describe_failure(row)

Питання: Для кого встановлюється стипендія ректора?
  правильний документ: ['344'], знайдено: ['344', '340', '344', '340', '340'] — ретрив знайшов
  правильно: False, цитування: True
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань. [1]

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  правильний документ: ['3320'], знайдено: ['3329', '3329', '3320', '3329', '3329'] — ретрив знайшов
  правильно: False, цитування: False
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].



Кожну неправильну відповідь відносять до однієї з двох груп:
- **помилка оцінювання**: відповідь правильна, але суддя помилився (відповідь сформульована інакше або містить більше деталей, ніж еталон)
  або еталон неоднозначний (відповідь спирається на інше місце документа, ніж еталон, і не суперечить документу);
- **помилка системи**: промах ретриву, галюцинація попри наявний контекст або суперечливі джерела —
  як у питанні про шифр спеціальності, де чинна редакція програми сама містить старий шифр 121 (див. демо 2).

Перед тим як змінювати систему, варто перевірити, чи правильно оцінено відповідь.

## Підсумок

- LangChain дає готові компоненти — розбивачі тексту, обгортки моделей, векторні сховища, ретривери — і спосіб
  зʼєднати їх у ланцюжок; логіку пошуку, як-от гібридний пошук чи переранжування, легко додати власними функціями.
- BM25 і семантичний пошук помиляються на різних питаннях, тому їхнє поєднання зазвичай краще за кожен окремо.
- Cross-encoder точніше оцінює відповідність фрагмента запиту, але працює лише на невеликій кількості кандидатів.
- Ретрив і генерацію оцінюють окремо: так видно, де саме виникає помилка.
- Для автоматичного оцінювання відповідей існують готові бібліотеки, наприклад [RAGAS](https://docs.ragas.io/);
  вони теж спираються на LLM як суддю.

## 0.1. Теоретичні питання 21–30

21. Виміряний час і стан кожної клітинки наведено в кінці.
22. RecursiveCharacterTextSplitter пробує розділювачі від більших одиниць до менших і пакує отримані частини. Демо 2 виділяє речення регулярним виразом. keep_separator="end" зберігає крапку чи інший роздільник у кінці попередньої частини.
23. Для одиничних векторів внутрішній добуток дорівнює косинусу. FAISS працює і без нормалізації, але тоді норма документа впливає на результат і порядок може змінитися.
24. Вивід додаткового бенчмарку дає приклади з перевагою кожного методу. Точні терміни й коди можуть допомогти BM25, а різні словоформи й перефразування — семантичному пошуку.
25. Ручне значення RRF обчислено як 1/(60+2) + 1/(60+5). Використання позицій дозволяє об'єднувати пошукачі з різними шкалами без калібрування.
26. Cross-encoder обробляє кожну пару «питання — фрагмент» разом. Ознаки фрагмента залежать від запиту. Повне переранжування потребує стільки пар, скільки всього фрагментів. Кількість фактичних проходів залежить від батчування.
27. Додано питання про фальсифікацію, перефразування про гуртожиток і запит про басейн. Останній виключено з hit@k і MRR, бо в нього немає еталонного доказу. Він входить в оцінку відмов. Порівняння в таблиці додаткового бенчмарку.
28. Cross-encoder може підняти докази з нижчих позицій серед 20 кандидатів. Це напряму впливає на hit@1. Якщо доказу немає серед кандидатів, переранжування його не відновить. Для оцінки ефекту порівнюю всі три метрики.
29. Спільна модель судді й генератора створює залежність помилок. Суддя може віддавати перевагу власним формулюванням або приймати правдоподібне твердження без достатнього доказу. Нижче наведено п'ять прикладів для ручної звірки.
30. Під час діагностики перевіряю еталонний уривок у топ-5, відповідь, цитату й вердикт. Збіг ідентифікатора документа без потрібного фрагмента сам по собі не гарантує успіх ретриву.

In [17]:
for index_row in [0, 2, 8, 11, len(BENCHMARK) - 1]:
    question, gold, evidence, reference = BENCHMARK[index_row]
    row = evaluation.iloc[index_row]
    print(f"{index_row + 1}. {question}\nЕталон: {reference or '(у корпусі немає відповіді)'}"
          f"\nМодель: {row['відповідь']}\nСуддя: {bool(row['правильно'])}. Цитата: {row['цитування']}\n")

for index_row, item in evaluation.iterrows():
    if item["правильно"] and item["цитування"] != False:
        continue
    question, gold, evidence, reference = BENCHMARK[index_row]
    relevant_rank = first_hit_rank(rerank_search(question, 5), gold, evidence) if gold else None
    print(f"№{index_row + 1} | ранг доказу: {relevant_rank} | оцінка судді: {item['правильно']}"
          f" | цитата: {item['цитування']}\n{question}\nЕталон: {reference}\n{item['відповідь']}\n")

1. До якої дати потрібно оплатити гуртожиток за перший семестр?
Еталон: До 1 вересня поточного року.
Модель: До 1 вересня поточного року потрібно оплатити гуртожиток за перший семестр [1].
Суддя: True. Цитата: True

3. З якої до якої години в гуртожитку має бути тиша?
Еталон: З 23:00 до 7:00.
Модель: З 23:00 години до 7:00 години в гуртожитку повинна додержуватись тиша [1].
Суддя: True. Цитата: True

9. На який строк обирається староста групи?
Еталон: На весь період навчання.
Модель: Староста обирається на весь період навчання [1].
Суддя: True. Цитата: True

12. Яку частку кредитів становлять вибіркові дисципліни?
Еталон: Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням).
Модель: Вибіркові дисципліни становлять не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням). [2]
Суддя: True. Цитата: True

29. Де знайти ціни на абонемент до університетс

№17 | ранг доказу: 1 | оцінка судді: False | цитата: True
Для кого встановлюється стипендія ректора?
Еталон: Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності.
Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань. [1]



№23 | ранг доказу: None | оцінка судді: False | цитата: False
Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
Еталон: F2.
Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].



## Фактичний запуск у Google Colab

Загальний час (з підготовкою ядра) — 330.2 с.

| Клітинка в ноутбуку | Час, с |
|---:|---:|
| 2 | 13.85 |
| 3 | 18.88 |
| 5 | 17.32 |
| 6 | 0.01 |
| 8 | 0.02 |
| 10 | 38.30 |
| 12 | 0.18 |
| 15 | 0.06 |
| 17 | 28.09 |
| 19 | 0.01 |
| 21 | 9.22 |
| 22 | 19.94 |
| 25 | 37.89 |
| 27 | 3.62 |
| 29 | 136.73 |
| 31 | 0.01 |
| 35 | 0.72 |

Стан запуску — виконано.

### Фактичні результати до питань 21, 24–30

Запуск зайняв 330,2 с. Pip повідомив про несумісність версій requests і huggingface-hub з іншими пакетами середовища. Клітинки цього ноутбука завершилися. Для повного переранжування було б 385 пар за запит замість 20, приблизно 19 разів більше. RRF для заданих позицій становить 0,03151365.

Для питання про медичну академвідпустку BM25 поставив фрагмент із доказом на 1-ше місце, векторний пошук — на 6-те. Запит має багато точних термінів. Для питання «Як часто виплачують стипендію?» векторний метод поставив правильний доказ на 1-ше місце, а BM25 не знайшов його в топ-10. Формулювання про періодичність має збігатися за змістом із «один раз на місяць», навіть коли немає спільних слів.

Після доповнення бенчмарку hit@1 векторного методу змінився з 0,609 до 0,600, BM25 з 0,522 до 0,480, гібридного пошуку з 0,609 до 0,600, а після переранжування — з 0,826 до 0,840. Підсумкові MRR — 0,726, 0,564, 0,736 і 0,905 відповідно. У розширеному бенчмарку приріст від переранжування поверх гібридного методу становив 0,24 для hit@1 і 0,08 для hit@5.

**Ручна звірка п'яти прикладів.** №1 передає дату 1 вересня. №3 правильно відтворює час 23:00–7:00. №9 правильно вказує весь період навчання. №12 зберігає і загальний мінімум 25%, і виняток 10%. №29 коректно відмовляється називати ціни басейну. У всіх п'яти випадках ручні вердикти збігаються з автоматичними. Для №29 автоматичним критерієм є фраза відмови.

**Два випадки помилки.**

- №17. Стипендія ректора. Доказ у топ-1, цитування зараховане. Пункт 1.1 положення прямо згадує переможців міжнародних та всеукраїнських змагань разом із суттєвими досягненнями. Модель передала обидві частини, а еталон містить лише першу. Суддя позначив відповідь як неправильну. За звіркою з документом це неповний еталон і хибнонегативний вердикт судді. [Положення про стипендію ректора, пункт 1.1](https://docs.ztu.edu.ua/?mdocs-file=344#page=3)
- №23. Шифр спеціальності. Потрібного фрагмента з F2 не було в топ-5. Модель відповіла 121 і послалася на інший фрагмент. Це промах ретриву з хибною відповіддю на застарілих або суперечливих даних. Варіанти виправлення — пріоритет редакції та окреме структурування шифру.